In [ ]:
from torch.utils.data import DataLoader
from PIL import Image
import os
import numpy as np
from glob import glob
from sklearn.model_selection import train_test_split
import sys
sys.path.append("..")  # aggiunge la cartella padre (project/) al path
from data.dataset import MayoDataset, MayoDatasetNpy
from data.preprocessing import get_transform
import matplotlib.pyplot as plt
import random

In [ ]:
print(os.getcwd())

# ============================================================
# CREATE DIRECTORY STRUCTURE
# ============================================================

BASE_DATA_PATH = "../data"

PREPROCESSED_PATH = os.path.join(BASE_DATA_PATH, "preprocessed")
SINOGRAM_CLEAN_PATH = os.path.join(BASE_DATA_PATH, "sinogram_clean")
SINOGRAM_CORRUPTED_PATH = os.path.join(BASE_DATA_PATH, "sinogram_corrupted")

splits = ["train", "validation", "test"]
angle_folders = ["angles_180", "angles_90", "angles_60", "angles_45"]

# ------------------------------------------------------------
# PREPROCESSED
# ------------------------------------------------------------
for split in splits:
    os.makedirs(
        os.path.join(PREPROCESSED_PATH, split),
        exist_ok=True
    )

# ------------------------------------------------------------
# SINOGRAMS
# ------------------------------------------------------------
for split in splits:

    for angle_folder in angle_folders:

        os.makedirs(
            os.path.join(
                SINOGRAM_CLEAN_PATH,
                split,
                angle_folder
            ),
            exist_ok=True
        )

        os.makedirs(
            os.path.join(
                SINOGRAM_CORRUPTED_PATH,
                split,
                angle_folder
            ),
            exist_ok=True
        )

print("Directory structure created.")
    
# ============================================================
# SAVE PREPROCESSED IMAGES
# ============================================================

def save_preprocessed_images(image_paths, split_name, transform):
    save_dir = os.path.join(PREPROCESSED_PATH, split_name)

    for img_path in image_paths:
        image = Image.open(img_path).convert("L")
        image_tensor = transform(image)
        image_np = image_tensor.squeeze().numpy()

        # crea nome univoco usando il path relativo rispetto a DATASET_PATH
        rel_path = os.path.relpath(img_path, DATASET_PATH)
        filename = rel_path.replace(os.sep, "_")
        filename = os.path.splitext(filename)[0] + ".npy"

        save_path = os.path.join(save_dir, filename)
        np.save(save_path, image_np)

    print(f"Saved {split_name}: {len(image_paths)} immagini")

DATASET_PATH = "../data/raw"

all_images = glob(
    os.path.join(DATASET_PATH, "**", "*.png"),
    recursive=True
)

print("Found images:", len(all_images))

train_images_raw, temp_images_raw = train_test_split(
    all_images,
    test_size=0.3,
    random_state=42
)

val_images_raw, test_images_raw = train_test_split(
    temp_images_raw,
    test_size=0.5,
    random_state=42
)

transform = get_transform()

# ============================================================
# SAVE PREPROCESSED DATASET
# ============================================================

save_preprocessed_images(train_images_raw, "train", transform)
save_preprocessed_images(val_images_raw, "validation", transform)
save_preprocessed_images(test_images_raw, "test", transform)


print(f"Totale immagini:    {len(all_images)}")
print(f"Train:              {len(train_images_raw)} ({len(train_images_raw
)/len(all_images)*100:.1f}%)")
print(f"Validation:         {len(val_images_raw)} ({len(val_images_raw)/len(all_images)*100:.1f}%)")
print(f"Test:               {len(test_images_raw)} ({len(test_images_raw)/len(all_images)*100:.1f}%)")

In [ ]:
PREPROCESSED_PATH = "../data/preprocessed"

train_images_npy = glob(os.path.join(PREPROCESSED_PATH, "train", "*.npy"))
val_images_npy   = glob(os.path.join(PREPROCESSED_PATH, "validation", "*.npy"))
test_images_npy  = glob(os.path.join(PREPROCESSED_PATH, "test", "*.npy"))

train_loader = DataLoader(MayoDatasetNpy(train_images_npy), batch_size=8, shuffle=True)
val_loader   = DataLoader(MayoDatasetNpy(val_images_npy),   batch_size=8, shuffle=False)
test_loader  = DataLoader(MayoDatasetNpy(test_images_npy),  batch_size=8, shuffle=False)

In [ ]:

def show_raw_vs_preprocessed(raw_path, transform, n=5):
    fig, axes = plt.subplots(2, n, figsize=(3*n, 6))
    
    sample_paths = random.sample(raw_path, n)
    
    for i, path in enumerate(sample_paths):
        # raw
        raw = Image.open(path).convert("L")
        axes[0, i].imshow(raw, cmap="gray")
        axes[0, i].set_title(f"Raw\n{np.array(raw).shape}\n[{np.array(raw).min()}, {np.array(raw).max()}]")
        axes[0, i].axis("off")
        
        # preprocessata
        tensor = transform(raw).squeeze().numpy()
        axes[1, i].imshow(tensor, cmap="gray")
        axes[1, i].set_title(f"Preprocessed\n{tensor.shape}\n[{tensor.min():.2f}, {tensor.max():.2f}]")
        axes[1, i].axis("off")
    
    plt.suptitle("Raw vs Preprocessed", fontsize=14)
    plt.tight_layout()
    plt.show()

show_raw_vs_preprocessed(train_images_raw, transform)

In [ ]:

# ============================================================
# CHECK 1: Distribuzione del dataset
# ============================================================
print("=== DATASET SPLIT ===")
print(f"Totale immagini:  {len(all_images)}")
print(f"Train:            {len(train_images_raw)} ({len(train_images_raw)/len(all_images)*100:.1f}%)")
print(f"Validation:       {len(val_images_raw)} ({len(val_images_raw)/len(all_images)*100:.1f}%)")
print(f"Test:             {len(test_images_raw)} ({len(test_images_raw)/len(all_images)*100:.1f}%)")

# ============================================================
# CHECK 2: Verifica file .npy salvati
# ============================================================
print("\n=== FILE .NPY SALVATI ===")
print(f"Train .npy:       {len(train_images_npy)}")
print(f"Validation .npy:  {len(val_images_npy)}")
print(f"Test .npy:        {len(test_images_npy)}")

# ============================================================
# CHECK 3: Shape, dtype e range di un campione .npy
# ============================================================
print("\n=== TYPE & SHAPE CHECK (.npy) ===")
sample_np = np.load(train_images_npy[0])
print(f"Tipo:             {type(sample_np)}")
print(f"Shape:            {sample_np.shape}")           # atteso: (256, 256)
print(f"Dtype:            {sample_np.dtype}")           # atteso: float32
print(f"Min value:        {sample_np.min():.4f}")       # atteso: >= 0.0
print(f"Max value:        {sample_np.max():.4f}")       # atteso: <= 1.0
print(f"Valori in [0,1]:  {(sample_np >= 0).all() and (sample_np <= 1).all()}")

# ============================================================
# CHECK 4: Verifica batch dal DataLoader
# ============================================================
print("\n=== BATCH CHECK ===")
batch = next(iter(train_loader))
print(f"Shape batch:      {batch.shape}")    # atteso: [8, 1, 256, 256]
print(f"Dtype batch:      {batch.dtype}")    # atteso: torch.float32
print(f"Min batch:        {batch.min():.4f}")
print(f"Max batch:        {batch.max():.4f}")

# ============================================================
# CHECK 5: Dimensioni originali (raw .png)
# ============================================================
print("\n=== DIMENSIONI ORIGINALI (prime 5 immagini raw) ===")
for path in train_images_raw[:5]:
    img = Image.open(path).convert("L")
    print(f"  {os.path.basename(path):40s} → {img.size[0]}x{img.size[1]} px")

# ============================================================
# CHECK 6: Visualizza raw vs preprocessata (.npy)
# ============================================================
# CHECK 6: Visualizza raw vs preprocessata (.npy)
n = 3
fig, axes = plt.subplots(n, 2, figsize=(8, 4*n))
fig.suptitle("Raw (.png) vs Preprocessata (.npy)", fontsize=14)

samples_raw = random.sample(train_images_raw, n)

for i, raw_path in enumerate(samples_raw):
    # ricostruisce il nome .npy con lo stesso metodo usato nel salvataggio
    rel_path = os.path.relpath(raw_path, DATASET_PATH)
    filename_npy = rel_path.replace(os.sep, "_")
    filename_npy = os.path.splitext(filename_npy)[0] + ".npy"
    npy_path = os.path.join(PREPROCESSED_PATH, "train", filename_npy)

    # raw
    img_raw = Image.open(raw_path).convert("L")
    axes[i, 0].imshow(img_raw, cmap="gray")
    axes[i, 0].set_title(f"Raw: {img_raw.size[0]}×{img_raw.size[1]} | [{np.array(img_raw).min()}, {np.array(img_raw).max()}]")
    axes[i, 0].axis("off")

    # preprocessata
    img_npy = np.load(npy_path)
    axes[i, 1].imshow(img_npy, cmap="gray")
    axes[i, 1].set_title(f"Preprocessata: 256×256 | [{img_npy.min():.2f}, {img_npy.max():.2f}]")
    axes[i, 1].axis("off")

plt.tight_layout()
plt.show()


In [ ]:
#This code is just the preliminary for sinogram creation

'''
from skimage.transform import radon, iradon
import matplotlib.pyplot as plt
import numpy as np

# ============================================================
# SAVE SINOGRAMS
# ============================================================

def save_sinograms(dataset, split_name):

    for idx in range(len(dataset)):

        img = dataset[idx].squeeze().numpy()

        for n_angles in ANGLE_CONFIGS:

            angles = np.linspace(
                0,
                180,
                n_angles,
                endpoint=False
            )

            # ------------------------------------------------
            # CLEAN SINOGRAM
            # ------------------------------------------------
            sinogram = radon(
                img,
                theta=angles,
                circle=True
            )

            # ------------------------------------------------
            # NOISY SINOGRAM
            # ------------------------------------------------
            noisy_sinogram = sinogram + np.random.normal(
                0,
                noise_level * sinogram.max(),
                sinogram.shape
            )

            # ------------------------------------------------
            # NORMALIZATION FOR SAVING
            # ------------------------------------------------
            sinogram_norm = (
                sinogram - sinogram.min()
            ) / (
                sinogram.max() - sinogram.min() + 1e-8
            )

            noisy_norm = (
                noisy_sinogram - noisy_sinogram.min()
            ) / (
                noisy_sinogram.max() - noisy_sinogram.min() + 1e-8
            )

            sinogram_uint8 = (
                sinogram_norm * 255
            ).astype(np.uint8)

            noisy_uint8 = (
                noisy_norm * 255
            ).astype(np.uint8)

            # ------------------------------------------------
            # SAVE PATHS
            # ------------------------------------------------
            clean_path = os.path.join(
                SINOGRAM_CLEAN_PATH,
                split_name,
                f"angles_{n_angles}",
                f"sino_{idx:05d}.png"
            )

            noisy_path = os.path.join(
                SINOGRAM_CORRUPTED_PATH,
                split_name,
                f"angles_{n_angles}",
                f"sino_{idx:05d}.png"
            )

            # ------------------------------------------------
            # SAVE
            # ------------------------------------------------
            Image.fromarray(sinogram_uint8).save(clean_path)

            Image.fromarray(noisy_uint8).save(noisy_path)

    print(f"Saved sinograms for {split_name}")

noise_level = 0.005
ANGLE_CONFIGS = [180, 90, 60, 45]

# ============================================================
# GENERATE AND SAVE SINOGRAMS
# ============================================================

save_sinograms(train_dataset, "train")
save_sinograms(val_dataset, "validation")
save_sinograms(test_dataset, "test")

'''

'''
for n_angles in ANGLE_CONFIGS:

    angles = np.linspace(0, 180, n_angles, endpoint=False)

    img_2d = train_dataset[0].squeeze().numpy()

    # =========================================================
    # FORWARD PROJECTION
    # =========================================================
    sinogram = radon(img_2d, theta=angles, circle=True)

    # =========================================================
    # ADD GAUSSIAN NOISE
    # =========================================================
    noisy_sinogram = sinogram + np.random.normal(
        0,
        noise_level * sinogram.max(),
        sinogram.shape
    )

    # =========================================================
    # FBP RECONSTRUCTION
    # =========================================================
    fbp = iradon(
        noisy_sinogram,
        theta=angles,
        filter_name='hamming',
        circle=True
    )

    fbp = np.clip(fbp, 0, 1) #clipping dovuto al fatto che alcuni valori sono minori di zero

    # =========================================================
    # HISTOGRAM DATA
    # =========================================================
    orig_pixels = img_2d.flatten()
    fbp_pixels = fbp.flatten()

    # =========================================================
    # PLOT
    # =========================================================
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))

    fig.suptitle(
        f"CT Reconstruction - n_angles = {n_angles}",
        fontsize=14
    )

    # ---------------------------------------------------------
    # ORIGINAL IMAGE
    # ---------------------------------------------------------
    axes[0, 0].set_title("Original CT")
    axes[0, 0].imshow(img_2d, cmap='gray')
    axes[0, 0].axis("off")

    # ---------------------------------------------------------
    # CLEAN SINOGRAM
    # ---------------------------------------------------------
    axes[0, 1].set_title("Sinogram (clean)")
    axes[0, 1].imshow(
        sinogram,
        cmap='gray',
        aspect='auto'
    )
    axes[0, 1].set_xlabel("Angle")

    # ---------------------------------------------------------
    # NOISY SINOGRAM
    # ---------------------------------------------------------
    axes[0, 2].set_title("Sinogram (noisy)")
    axes[0, 2].imshow(
        noisy_sinogram,
        cmap='gray',
        aspect='auto'
    )
    axes[0, 2].set_xlabel("Angle")

    # ---------------------------------------------------------
    # FBP RECONSTRUCTION
    # ---------------------------------------------------------
    axes[1, 0].set_title("FBP reconstruction")
    axes[1, 0].imshow(fbp, cmap='gray')
    axes[1, 0].axis("off")

    # ---------------------------------------------------------
    # HISTOGRAM ORIGINAL
    # ---------------------------------------------------------
    axes[1, 1].hist(
        orig_pixels,
        bins=60
    )

    axes[1, 1].set_title("Histogram - Original")
    axes[1, 1].set_xlabel("Pixel value")
    axes[1, 1].set_ylabel("Frequency")

    # ---------------------------------------------------------
    # HISTOGRAM FBP
    # ---------------------------------------------------------
    axes[1, 2].hist(
        fbp_pixels,
        bins=60
    )

    axes[1, 2].set_title("Histogram - FBP")
    axes[1, 2].set_xlabel("Pixel value")
    axes[1, 2].set_ylabel("Frequency")

    plt.tight_layout()
    plt.show()



# ============================================================
# FILTER COMPARISON
# ============================================================
noise_level = 0.005
n_angles = 180
filters = [
    'ramp',
    'shepp-logan',
    'cosine',
    'hamming',
    'hann'
]

angles = np.linspace(0, 180, n_angles, endpoint=False)

img_2d = train_dataset[0].squeeze().numpy()

sinogram = radon(
    img_2d,
    theta=angles,
    circle=True
)

noisy_sinogram = sinogram + np.random.normal(
    0,
    noise_level * sinogram.max(),
    sinogram.shape
)

fig, axes = plt.subplots(2, 3, figsize=(14, 9))

fig.suptitle(
    f"FBP Filter Comparison ({n_angles} angles)",
    fontsize=15
)

# ------------------------------------------------------------
# ORIGINAL IMAGE
# ------------------------------------------------------------
axes[0, 0].imshow(img_2d, cmap='gray')
axes[0, 0].set_title("Original CT")
axes[0, 0].axis("off")

# ------------------------------------------------------------
# NOISY SINOGRAM
# ------------------------------------------------------------
axes[0, 1].imshow(
    noisy_sinogram,
    cmap='gray',
    aspect='auto'
)

axes[0, 1].set_title("Noisy sinogram")
axes[0, 1].set_xlabel("Angle")

# ------------------------------------------------------------
# EMPTY PANEL
# ------------------------------------------------------------
axes[0, 2].axis("off")

# ============================================================
# RECONSTRUCTIONS
# ============================================================
positions = [
    (1, 0),
    (1, 1),
    (1, 2)
]

# first 3 filters
for idx, filt in enumerate(filters[:3]):

    row, col = positions[idx]

    recon = iradon(
        noisy_sinogram,
        theta=angles,
        filter_name=filt,
        circle=True
    )

    recon = np.clip(recon, 0, 1)

    axes[row, col].imshow(recon, cmap='gray')
    axes[row, col].set_title(f"{filt}")
    axes[row, col].axis("off")

plt.tight_layout()
plt.show()
'''

# Sinogram creation with the IPPy package
'''
from ippy import operators, solvers
from ippy.metrics import SSIM, PSNR, RE
import torch

noise_level = 0.005
ANGLE_CONFIGS = [180, 90, 60, 45]

for n_angles in ANGLE_CONFIGS:

    angles = np.linspace(0, np.pi, n_angles, endpoint=False)
    img_2d = train_dataset[0].unsqueeze(0)

    print(f"Dimensioni di img_2d: {img_2d.shape}")
    print(f"Numero totale di elementi: {img_2d.numel()}") # Se è un tensore PyTorch
    
    K = operators.CTProjector(
        img_shape = (256,256),
        angles = angles,
        det_size= 256, 
        geometry = "parallel"
    )

    sinogram = K(img_2d)
    ampiezza_rumore = noise_level * sinogram.max()
    sinogram_noisy = sinogram + ampiezza_rumore * torch.randn_like(sinogram)

    solver1 = solvers.FBP(K)

    x_sol, info = solver1(
        sinogram_noisy,
        x_true=img_2d,
        starting_point=sinogram_noisy
    )

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    fig.suptitle(f"n_angles = {n_angles}", fontsize=13)

    img_2d = img_2d.squeeze()

    axes[0].set_title("Original")
    axes[0].imshow(img_2d, cmap=plt.cm.Greys_r)
    axes[0].axis("off")

    dx = 0.5 * 180.0 / max(img_2d.shape)
    dy = 0.5 / sinogram.shape[0]
    extent = (-dx, 180.0 + dx, -dy, sinogram.shape[0] + dy)

    axes[1].set_title("Sinogram (clean)")
    axes[1].imshow(sinogram.squeeze(), cmap=plt.cm.Greys_r, extent=extent, aspect='auto')
    axes[1].set_xlabel("Angle (deg)")

    axes[2].set_title("Sinogram (noisy)")
    axes[2].imshow(sinogram_noisy.squeeze(), cmap=plt.cm.Greys_r, extent=extent, aspect='auto')
    axes[2].set_xlabel("Angle (deg)")

    axes[3].set_title("FBP reconstruction")
    axes[3].imshow(x_sol.squeeze(), cmap=plt.cm.Greys_r)
    axes[3].axis("off")

    fig.tight_layout()
    plt.show()

'''
